# Echo Protocol Preprocessing Notebook

Parameterized notebook for generating Beckman Echo liquid handler protocols.
Accepts parameters via papermill for custom experiment configuration.

In [ ]:
# Parameters passed by papermill

In [ ]:
workflow_name = "echo-protocol-preprocessing"
plate_mode = "Create a new plate from scratch"
premade_plate_name = None
new_plate_label = "new_source_plate"
experiment_name = "my_experiment"
doses = 6
doses2 = 0
highest_dose = 4
vol_cellextract = 2000
vol_antigen = 2000
samples = ["sample1", "sample2", "sample3"]
output_dir = "./generated"


In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import re

slug_pattern = re.compile(r"[^a-z0-9]+")
def slugify(name):
    return slug_pattern.sub("-", name.strip().lower()).strip("-")

In [ ]:
# Define plate geometry
plate_columns = list(range(1, 25))
plate_rows = list("ABCDEFGHIJKLMNOP")
wells_384 = [f"{row}{col}" for row in plate_rows for col in plate_columns]
wells_96 = [f"{row}{col}" for row in plate_rows[:8] for col in plate_columns[:12]]

print(f"384-well plate: {len(wells_384)} wells")
print(f"96-well plate: {len(wells_96)} wells")
print(f"Samples: {samples}")
print(f"Doses: {doses}, Second curve: {doses2}")

In [ ]:
# Calculate dose volumes (approximately 1:3 serial dilutions)
def calculate_dose_volumes(n_doses, max_volume_nl, dilution_factor=3.0):
    """Calculate volumes for serial dilution."""
    if n_doses == 0:
        return []
    volumes = []
    for i in range(n_doses):
        volume = int(max_volume_nl / (dilution_factor ** i))
        volumes.append(max(volume, 10))
    return volumes

antigen1_volumes = calculate_dose_volumes(doses, vol_antigen)
antigen1_pbs = [vol_antigen - v for v in antigen1_volumes]

antigen2_volumes = calculate_dose_volumes(doses2, vol_antigen)
antigen2_pbs = [vol_antigen - v for v in antigen2_volumes]

print(f"Antigen 1 volumes: {antigen1_volumes} nL")
print(f"Antigen 1 PBS: {antigen1_pbs} nL")
print(f"Antigen 2 volumes: {antigen2_volumes} nL")
print(f"Antigen 2 PBS: {antigen2_pbs} nL")

In [ ]:
# Create source plate data structures
source_sample_names = ["NaN"] * 384
source_sample_vol = [0] * 384

sample_well_capacity = 60000

sample_wells = []
for i, sample in enumerate(samples):
    if i < 96:
        well_idx = i * 2
        if well_idx < 192:
            source_sample_names[well_idx] = sample
            source_sample_vol[well_idx] = sample_well_capacity
            sample_wells.append(wells_384[well_idx])

print(f"Sample wells: {len(sample_wells)} allocated")

In [ ]:
# Calculate reagent requirements
total_wells_needed = len(samples) * (doses + doses2)

total_antigen1_vol = sum(antigen1_volumes) * len(samples)
antigen_well_capacity = 65000
antigen1_wells_needed = (total_antigen1_vol // antigen_well_capacity) + 1

total_antigen2_vol = sum(antigen2_volumes) * len(samples)
antigen2_wells_needed = (total_antigen2_vol // antigen_well_capacity) + 1 if doses2 > 0 else 0

total_pbs1_vol = sum(antigen1_pbs) * len(samples)
total_pbs2_vol = sum(antigen2_pbs) * len(samples)
pbs_wells_needed = ((total_pbs1_vol + total_pbs2_vol) // antigen_well_capacity) + 1

print(f"Total destination wells: {total_wells_needed}")
print(f"Antigen 1 wells needed: {antigen1_wells_needed}")
print(f"Antigen 2 wells needed: {antigen2_wells_needed}")
print(f"PBS wells needed: {pbs_wells_needed}")

In [ ]:
# Place reagents in second half of source plate
reagent_start_idx = 192
reagent_idx = reagent_start_idx

antigen1_wells = []
for w in range(antigen1_wells_needed):
    if reagent_idx < 384:
        source_sample_names[reagent_idx] = "antigen1"
        remaining = total_antigen1_vol - (w * antigen_well_capacity)
        vol = min(antigen_well_capacity, remaining)
        source_sample_vol[reagent_idx] = max(vol, 1000)
        antigen1_wells.append(wells_384[reagent_idx])
        reagent_idx += 1

antigen2_wells = []
for w in range(antigen2_wells_needed):
    if reagent_idx < 384:
        source_sample_names[reagent_idx] = "antigen2"
        remaining = total_antigen2_vol - (w * antigen_well_capacity)
        vol = min(antigen_well_capacity, remaining)
        source_sample_vol[reagent_idx] = max(vol, 1000)
        antigen2_wells.append(wells_384[reagent_idx])
        reagent_idx += 1

pbs_wells = []
total_pbs = total_pbs1_vol + total_pbs2_vol
for w in range(pbs_wells_needed):
    if reagent_idx < 384:
        source_sample_names[reagent_idx] = "PBS"
        remaining = total_pbs - (w * antigen_well_capacity)
        vol = min(antigen_well_capacity, remaining)
        source_sample_vol[reagent_idx] = max(vol, 1000)
        pbs_wells.append(wells_384[reagent_idx])
        reagent_idx += 1

print(f"Antigen 1 source wells: {antigen1_wells}")
print(f"Antigen 2 source wells: {antigen2_wells}")
print(f"PBS source wells: {pbs_wells}")

In [ ]:
# Generate destination plate layout and protocol
dest_sample_names = []
dest_source_well = []
dest_dest_well = []
dest_transfer_vol = []

dest_well_idx = 0

for sample_idx, sample in enumerate(samples):
    if dest_well_idx >= 96:
        break
    
    source_well = sample_wells[sample_idx] if sample_idx < len(sample_wells) else sample_wells[0]
    
    for dose_idx in range(doses):
        if dest_well_idx >= 96:
            break
        
        dest_well = wells_96[dest_well_idx]
        
        dest_sample_names.append(sample)
        dest_source_well.append(source_well)
        dest_dest_well.append(dest_well)
        dest_transfer_vol.append(vol_cellextract)
        dest_well_idx += 1
        
        if antigen1_wells and antigen1_volumes[dose_idx] > 0:
            dest_sample_names.append(sample)
            dest_source_well.append(antigen1_wells[0])
            dest_dest_well.append(dest_well)
            dest_transfer_vol.append(antigen1_volumes[dose_idx])
        
        if antigen1_pbs[dose_idx] > 0 and pbs_wells:
            dest_sample_names.append(sample)
            dest_source_well.append(pbs_wells[0])
            dest_dest_well.append(dest_well)
            dest_transfer_vol.append(antigen1_pbs[dose_idx])

print(f"Total transfers generated: {len(dest_sample_names)}")

In [ ]:
# Create Echo protocol DataFrame
protocol_data = {
    "Sample Name": dest_sample_names,
    "Source Plate Name": ["source_plate"] * len(dest_sample_names),
    "Source Well": dest_source_well,
    "Destination Well": dest_dest_well,
    "Transfer Volume": dest_transfer_vol,
    "Destination Plate Name": ["dest_plate"] * len(dest_sample_names),
    "Source Plate Type": ["384PP_AQ_BP"] * len(dest_sample_names),
}

protocol_df = pd.DataFrame(protocol_data)
print(f"Total transfers: {len(protocol_df)}")

In [ ]:
# Save source plate and protocol CSVs
os.makedirs(output_dir, exist_ok=True)
experiment_slug = slugify(experiment_name)

source_plate_path = os.path.join(output_dir, f"{experiment_slug}_source_plate.csv")
protocol_path = os.path.join(output_dir, f"{experiment_slug}_echo_protocol.csv")

def make_plate():
    return pd.DataFrame(index=plate_rows, columns=plate_columns)

source_plate_df = make_plate()

for well_idx in range(384):
    if source_sample_vol[well_idx] > 0:
        row = well_idx // 24
        col = well_idx % 24
        source_plate_df.iat[row, col] = f"{source_sample_names[well_idx]}: {source_sample_vol[well_idx]}nL"

source_plate_df.to_csv(source_plate_path)
protocol_df.to_csv(protocol_path, index=False)

print(f"Source plate saved to: {source_plate_path}")
print(f"Protocol saved to: {protocol_path}")
print(f"Total transfers: {len(protocol_df)}")

In [ ]:
# Print summary
print("\n=== Protocol Summary ===")
print(f"Experiment: {experiment_name}")
print(f"Samples: {len(samples)}")
print(f"Doses per sample: {doses}")
print(f"Second curve doses: {doses2}")
print(f"Source plate wells used: {sum(1 for v in source_sample_vol if v > 0)}")
print(f"Total transfers: {len(protocol_df)}")
print(f"Total volume: {protocol_df['Transfer Volume'].sum()} nL")

In [ ]:
# Preview protocol
protocol_df.head(20)

In [ ]:
# Preview source plate
source_plate_df